# BDD100K Multi-Label Scene Classification — Exploration Notebook

This notebook walks through:
1. Parsing BDD100K JSON annotations
2. Visualising label distributions
3. Loading and displaying sample images
4. Running inference with a trained checkpoint
5. Generating Grad-CAM heatmaps

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'vision'))

import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

from dataset import parse_bdd100k_json, BDD100KDataset, LABEL_NAMES, get_val_transforms
from model import BDD100KClassifier
from gradcam import GradCAM, overlay_heatmap, denormalize
from metrics import per_label_f1, exact_match_ratio, hamming_loss, binarize

## 1  Configuration — set your paths here

In [ ]:
# ── Edit these to point at your BDD100K download ──
TRAIN_JSON  = "../data/bdd100k_labels_images_train.json"
VAL_JSON    = "../data/bdd100k_labels_images_val.json"
TRAIN_IMGS  = "../data/bdd100k/images/100k/train"
VAL_IMGS    = "../data/bdd100k/images/100k/val"
CHECKPOINT  = "../models/best_model.pth"

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {DEVICE}')

## 2  Parse annotations and inspect label distribution

In [ ]:
label_map = parse_bdd100k_json(TRAIN_JSON)
print(f'Annotations parsed: {len(label_map)}')

arr = np.stack(list(label_map.values()))   # (N, 4)
pos_rate = arr.mean(axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(LABEL_NAMES, pos_rate, color=['#e74c3c','#2c3e50','#3498db','#f39c12'])
ax.set_ylabel('Positive rate')
ax.set_title('BDD100K label distribution (train)')
for b, v in zip(bars, pos_rate):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.2%}', ha='center')
plt.tight_layout()
plt.show()

## 3  Sample images from each weather/time-of-day combination

In [ ]:
from pathlib import Path

ds = BDD100KDataset(
    image_dir=TRAIN_IMGS,
    json_path=TRAIN_JSON,
    transform=None,
    max_samples=500,
)

# Collect one sample per unique label vector
seen, samples = set(), []
for i in range(len(ds)):
    img_raw = Image.open(ds.image_paths[i]).convert('RGB')
    lbl = tuple(ds.label_vectors[i].astype(int))
    if lbl not in seen:
        seen.add(lbl)
        samples.append((img_raw, lbl))
    if len(samples) >= 8:
        break

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (img, lbl) in zip(axes.flat, samples):
    ax.imshow(img)
    label_str = ' | '.join(n for n, v in zip(LABEL_NAMES, lbl) if v)
    ax.set_title(label_str or '(no active label)', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample images — BDD100K', fontsize=13)
plt.tight_layout()
plt.show()

## 4  Load trained model and run batch inference on val set

In [ ]:
from metrics import MetricTracker
from torch.utils.data import DataLoader

model = BDD100KClassifier(pretrained=False)
ckpt  = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval().to(DEVICE)
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")

val_ds = BDD100KDataset(
    image_dir=VAL_IMGS,
    json_path=VAL_JSON,
    transform=get_val_transforms(),
    max_samples=300,
)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

tracker = MetricTracker()
with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(DEVICE))
        tracker.update(logits, labels)

metrics = tracker.compute()
print('\n--- Val metrics ---')
for k, v in metrics.items():
    print(f'  {k:<20}: {v:.4f}')

## 5  Grad-CAM heatmaps

In [ ]:
from gradcam import GradCAM, overlay_heatmap, denormalize
import cv2

grad_cam = GradCAM(model, DEVICE)
transform = get_val_transforms()

# Pick a few validation images
sample_paths = val_ds.image_paths[:4]

fig, axes = plt.subplots(len(sample_paths), len(LABEL_NAMES) + 1,
                          figsize=(5 * (len(LABEL_NAMES) + 1), 4 * len(sample_paths)))

for row, img_path in enumerate(sample_paths):
    pil_img = Image.open(img_path).convert('RGB')
    tensor  = transform(pil_img).unsqueeze(0)
    orig    = denormalize(tensor)

    axes[row, 0].imshow(orig)
    axes[row, 0].set_title('Original', fontsize=9)
    axes[row, 0].axis('off')

    with torch.no_grad():
        probs = torch.sigmoid(model(tensor.to(DEVICE))).squeeze().cpu().tolist()

    for col, (label, prob) in enumerate(zip(LABEL_NAMES, probs), start=1):
        heatmap = grad_cam.generate(tensor.clone(), label_idx=col - 1)
        overlay = overlay_heatmap(orig.copy(), heatmap)
        axes[row, col].imshow(overlay)
        axes[row, col].set_title(f'{label}\np={prob:.2f}', fontsize=9)
        axes[row, col].axis('off')

plt.suptitle('Grad-CAM heatmaps per label', fontsize=13)
plt.tight_layout()
plt.show()